# Day 13 — Build Your First AI-Powered CLI Tool

Build a CLI tool that takes a messy block of text and returns clean, structured JSON
(for example, pull name, email, and company out of an email signature). Add one tool
the model can call.

In [2]:
#Intializing the OpenAI Client

from openai import OpenAI
import os
from dotenv import load_dotenv
import json

load_dotenv()

client=OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [3]:
#Defining the JSON schema

contact_schema = {
    "type": "object",
    "properties": {
        "name": {
            "type": ["string", "null"]
        },
        "email": {
            "type": ["string", "null"]
        },
        "company": {
            "type": ["string", "null"]
        }
    },
    "required": [
        "name",
        "email",
        "company"
    ],
    "additionalProperties": False
}

In [4]:
#Contract extraction function - Which will connect the schma to the OpenAI API

def extract_contact(text):

    response = client.responses.create(
        model="gpt-4.1-mini",

        instructions="""
        Extract the person's name, email address,
        and company from the provided text.

        If information is missing, return null.
        Never invent information.
        """,

        input=text,

        text={
            "format": {
                "type": "json_schema",
                "name": "contact_extraction",
                "strict": True,
                "schema": contact_schema
            }
        }
    )

    contact = json.loads(response.output_text)

    return contact

In [5]:
#Testing your function
sample_text = """
Hello!

My name is David Miller.
I work at Microsoft.
My email is david@microsoft.com.

Thank you!
"""

result = extract_contact(sample_text)

print(result)

{'name': 'David Miller', 'email': 'david@microsoft.com', 'company': 'Microsoft'}


In [6]:
#Adding a callable python tool
def count_words(text):
    words = text.split()
    return len(words)

In [7]:
#Now describing the function back to the model

tools = [
    {
        "type": "function",
        "name": "count_words",
        "description": "Count the number of words in a text.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string"
                }
            },
            "required": ["text"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [8]:
#ASking the model to use  the tool

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Count the words in this sentence: AI engineering is exciting.",
    tools=tools,
    tool_choice="required"
)

In [11]:
#inspecting the result
for item in response.output:

    if item.type == "function_call":

        print("Tool:", item.name)
        print("Arguments:", item.arguments)
        print("Call ID:", item.call_id)

#Model willl request a toold and the python fucntion executes it.

Tool: count_words
Arguments: {"text":"AI engineering is exciting."}
Call ID: call_PqzMToC7vYyh9rdEmoBI3ucB


In [12]:
for item in response.output:

    if item.type == "function_call":

        arguments = json.loads(item.arguments)

        result = count_words(**arguments)

        print("Word count:", result)

Word count: 4
